# 2 · make — contact-prediction accuracy data

Aggregates [#245](https://github.com/Open-Athena/MarinFold/issues/245)'s published per-protein
R-precision into the table [`2_plot_rprecision.ipynb`](2_plot_rprecision.ipynb) draws. Nothing is
re-scored here — these are the numbers of record, read from the public bucket.

**Two protein classes, and why these ones.** *Natural* is the 314 natural FoldBench monomers
(`eval-val` + `eval-test`). *Designed* is FoldBench's 19 de novo monomers (`eval-denovo`) — the
only designed set where a baseline comparison is legitimate, because exp65's 396 designs are 20x
larger but 50.5 % of them were deposited on or before Protenix-v2's 2021-09-30 training cutoff,
which makes the baselines the contaminated party there.

`eval-test` is a held-out set with a read budget
([`eval_test_reads.md`](../../experiments/exp245_evals_foldbench_held_out_monomers/data/eval_test_reads.md)).
Re-displaying its published numbers is not a new read; scoring something new on it is.

CPU only.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "2_rprecision"
RANGE = "all"        # separation range: all | short | medium | long
METRIC = "R"         # R-precision; L, L/2, L/5 and AUC are also published
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
CLASSES = {          # name -> the eval sets it pools
    "natural": ["eval-val", "eval-test"],
    "designed": ["eval-denovo"],
}
PARAMETERS = dict(range=RANGE, metric=METRIC, classes=CLASSES,
                  bootstrap_draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED)
PARAMETERS

In [ ]:
import pandas as pd

inputs = figlib.Inputs()
targets = figlib.load_foldbench_universe(inputs)
scores = figlib.load_foldbench_scores(inputs)
scores = scores[(scores.range == RANGE) & (scores.cut == METRIC)]
print(f"{len(targets)} units · {scores.predictor.nunique()} predictors")
print(targets.eval_set.value_counts().to_string())

In [ ]:
rows, per_protein = [], []
for class_name, eval_sets in CLASSES.items():
    units = targets[targets.eval_set.isin(eval_sets)]
    keys = set(zip(units.dataset, units.stem))
    subset = scores[[key in keys for key in zip(scores.dataset, scores.stem)]]
    per_protein.append(subset.assign(protein_class=class_name))
    for predictor, group in subset.groupby("predictor"):
        mean, low, high = figlib.bootstrap_mean(group.value.values, BOOTSTRAP_DRAWS,
                                                BOOTSTRAP_SEED)
        rows.append(dict(protein_class=class_name, eval_sets="+".join(eval_sets),
                         predictor=predictor, n=len(group), value=mean,
                         ci_low=low, ci_high=high))

summary = pd.DataFrame(rows).sort_values(["protein_class", "value"], ascending=[True, False])
per_protein = pd.concat(per_protein, ignore_index=True)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
figlib.write_dataset(
    DATASET,
    notebook="2_make_rprecision_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "summary.csv": lambda path: summary.to_csv(path, index=False),
        "per_protein.csv": lambda path: per_protein.to_csv(path, index=False),
    },
    extra={
        "metric": {"range": RANGE, "cut": METRIC,
                   "definition": "fraction of the N highest-confidence predicted contacts that "
                                 "are observed, where N is the protein's observed contact count"},
        "classes": {name: int((per_protein.protein_class == name).sum() //
                              max(1, per_protein[per_protein.protein_class == name]
                                  .predictor.nunique()))
                    for name in CLASSES},
    })